In [51]:
import pandas as pd
import numpy as np
import seaborn as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV,RandomizedSearchCV,train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score,classification_report,average_precision_score,confusion_matrix,roc_auc_score,roc_curve,precision_recall_curve)

from category_encoders import TargetEncoder
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical


In [52]:
df = pd.read_csv("heart_original.csv")
df = df.dropna()
df = df[(df != '').all(axis=1)]

print(df.columns.tolist())
print()
print(df.shape)
print()
print(df.head())
print()
print(df.isnull().sum())


['Age', 'Sex', 'ChestPain', 'RestBP', 'Chol', 'Fbs', 'RestECG', 'MaxHR', 'ExAng', 'Oldpeak', 'Slope', 'Ca', 'Thal', 'Target']

(301, 14)

   Age  Sex     ChestPain  RestBP  Chol  Fbs  RestECG  MaxHR  ExAng  Oldpeak  \
0   63    1       typical     145   233    1        2    150      0      2.3   
1   67    1  asymptomatic     160   286    0        2    108      1      1.5   
2   67    1  asymptomatic     120   229    0        2    129      1      2.6   
3   37    1    nonanginal     130   250    0        0    187      0      3.5   
4   41    0    nontypical     130   204    0        2    172      0      1.4   

   Slope  Ca        Thal  Target  
0      3   0       fixed       0  
1      2   3      normal       1  
2      2   2  reversable       1  
3      3   0      normal       0  
4      1   0      normal       0  

Age          0
Sex          0
ChestPain    0
RestBP       0
Chol         0
Fbs          0
RestECG      0
MaxHR        0
ExAng        0
Oldpeak      0
Slope        0
Ca   

In [53]:
# defining numerical and categorical columns
cat_columns  = ['ChestPain','Thal']
num_columns  = ['Age', 'Sex','RestBP', 'Chol', 'Fbs', 'RestECG', 'MaxHR', 'ExAng', 'Oldpeak', 'Slope', 'Ca']

x = df.drop('Target',axis =1)
y = df['Target']
#split the data and print the percentage of Target in each set
x_train,x_test,y_train,y_test = train_test_split(
    x,y, test_size=0.2 , random_state=42, stratify=y
)
print(x_train.shape)
print(x_test.shape)
print(y_train.value_counts(normalize=True))
print()
print(y_test.value_counts(normalize=True))




(240, 13)
(61, 13)
Target
0    0.541667
1    0.458333
Name: proportion, dtype: float64

Target
0    0.540984
1    0.459016
Name: proportion, dtype: float64


In [54]:
#One hot encoding with sklearn instead of pandas dummies
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_columns), ("cat", OneHotEncoder(drop = "first",handle_unknown="ignore"), cat_columns)
    ]
)


# Fit on training data ONLY to avoid data leakage
x_train_array = preprocessor.fit_transform(x_train)
x_test_array = preprocessor.transform(x_test)

#convert to dataframe for easier visual and manipulation
# 🔹 STEP 1 — Get numeric column names (same as original)
num_feature_names = num_columns

# 🔹 STEP 2 — Get the one-hot encoded column names
ohe = preprocessor.named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(cat_columns)

# 🔹 Combine numeric + categorical encoded names
all_feature_names = list(num_feature_names) + list(cat_feature_names)

# -------------------------------------------------------------
# 🔹 STEP 3 — Convert to DataFrame with proper column names
x_train_oh = pd.DataFrame(x_train_array, columns=all_feature_names, index=x_train.index)
x_test_oh = pd.DataFrame(x_test_array, columns=all_feature_names, index=x_test.index)

# -------------------------------------------------------------
print("TRAIN:")
print(x_train_oh.head())

print("\nTEST:")
print(x_test_oh.head())

print("\nShapes:")
print("Train shape:", x_train_oh.shape)
print("Test shape:", x_test_oh.shape)


TRAIN:
          Age       Sex    RestBP      Chol       Fbs   RestECG     MaxHR  \
55  -0.058472  0.661155 -0.379835  0.326642 -0.420084  1.031758 -1.781425   
210 -1.951854 -1.512505 -0.605061 -0.636987 -0.420084 -0.973278  0.898401   
169 -1.060851 -1.512505 -1.055514 -1.676195 -0.420084 -0.973278 -0.507410   
112 -0.281223  0.661155 -0.717674 -1.184933 -0.420084  1.031758  1.777032   
184  0.609780 -1.512505  1.534589  1.063535 -0.420084  1.031758  0.503017   

        ExAng   Oldpeak     Slope        Ca  ChestPain_nonanginal  \
55   1.414214  1.030132  0.665660  0.410231                   0.0   
210 -0.707107 -0.883696 -0.948061 -0.708580                   1.0   
169 -0.707107 -0.883696  0.665660 -0.708580                   0.0   
112 -0.707107 -0.883696  0.665660 -0.708580                   0.0   
184 -0.707107 -0.883696 -0.948061 -0.708580                   0.0   

     ChestPain_nontypical  ChestPain_typical  Thal_normal  Thal_reversable  
55                    0.0             

In [58]:

# Create a copy of the data for Label Encoding
X_train_le = x_train.copy()
X_test_le = x_test.copy()

# Apply Label Encoding to categorical columns
label_encoders = {}
for col in cat_columns:
    le = LabelEncoder()
    X_train_le[col] = le.fit_transform(X_train_le[col].astype(str))
    X_test_le[col] = le.transform(X_test_le[col].astype(str))
    label_encoders[col] = le

# Scale numerical features
scaler_le = StandardScaler()
X_train_le[num_columns] = scaler_le.fit_transform(X_train_le[num_columns])
X_test_le[num_columns] = scaler_le.transform(X_test_le[num_columns])

print(f"After Label Encoding - Training shape: {X_train_le.shape}")

After Label Encoding - Training shape: (240, 13)


In [61]:

# Create a copy of the data for Target Encoding
X_train_te = x_train.copy()
X_test_te = x_test.copy()

# Apply Target Encoding
target_encoder = TargetEncoder(cols=cat_columns)
X_train_te = target_encoder.fit_transform(X_train_te, y_train)
X_test_te = target_encoder.transform(X_test_te)

# Scale numerical features
scaler_te = StandardScaler()
X_train_te[num_columns] = scaler_te.fit_transform(X_train_te[num_columns])
X_test_te[num_columns] = scaler_te.transform(X_test_te[num_columns])

print(f"After Target Encoding - Training shape: {X_train_te.shape}")

After Target Encoding - Training shape: (240, 13)



GRID SEARCH - One-Hot Encoding
Best parameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}

One-Hot + GridSearch Results:
Accuracy: 0.6885
ROC-AUC: 0.6991
Average Precision: 0.6016
Recall: 0.6786
Precision: 0.6552
F1-Score: 0.6667

GRID SEARCH - Label Encoding
Best parameters: {'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 4, 'min_samples_split': 2}

Label + GridSearch Results:
Accuracy: 0.7869
ROC-AUC: 0.8750
Average Precision: 0.8425
Recall: 0.6786
Precision: 0.8261
F1-Score: 0.7451

GRID SEARCH - Target Encoding
Best parameters: {'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}

Target + GridSearch Results:
Accuracy: 0.8033
ROC-AUC: 0.8728
Average Precision: 0.8171
Recall: 0.7857
Precision: 0.7857
F1-Score: 0.7857
